# Phase 7 EDA — Multi-Timeframe Strat-Sequence Dataset

Exploratory data analysis of the per-TF `strat_features_<tf>` tables built
by `gcp/research/p7_build_multi_tf_features.py` and analyzed by
`gcp/research/p7_analyze_tf.py`.

**Run order:**
1. Cell 1 — install deps + imports
2. Cell 2 — pull from GCS the per-TF analysis CSVs (must exist first)
3. Cell 3 onward — exploratory visualizations

**Source of truth for the raw data**: Cloud SQL tables
`strat_features_{1m,5m,15m,30m,60m}` (built by the Cloud Run Job).
This notebook pulls **only the aggregated analysis artifacts** from
GCS (small CSVs) — full row-level data is too large for in-notebook work.

In [ ]:
# Cell 1 — deps + imports
import subprocess, sys
for pkg in ['google-cloud-storage', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'pyarrow']:
    try:
        __import__(pkg.replace('-','_').replace('google_cloud_storage','google.cloud.storage'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import storage as gcs
import os

BUCKET = os.environ.get('GCS_BUCKET', 'adept-mountain-474619-d4-trading-data')
PREFIX = 'research/p7-analysis'
TFS = ['1m', '5m', '15m', '30m', '60m']

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print(f'Bucket: {BUCKET}')
print(f'Prefix: gs://{BUCKET}/{PREFIX}')

In [ ]:
# Cell 2 — pull all analysis CSVs from GCS into a dict-of-DataFrames
client = gcs.Client()
bucket = client.bucket(BUCKET)

def load_tf(tf):
    out = {}
    for fname in ['01_strat_transition.csv', '02_combo_dealer_regime.csv',
                  '03a_combo_vix.csv', '03b_combo_gex.csv', '03c_combo_vex.csv',
                  '04_indicator_correlations.csv',
                  '05a_model_walkforward.csv', '05b_model_summary.csv',
                  '05c_feature_importance_top50.csv']:
        blob = bucket.blob(f'{PREFIX}/{tf}/{fname}')
        try:
            data = blob.download_as_text()
            from io import StringIO
            out[fname] = pd.read_csv(StringIO(data))
        except Exception as e:
            print(f'  {tf}/{fname}: not yet — {e.__class__.__name__}')
    return out

data = {tf: load_tf(tf) for tf in TFS}
for tf in TFS:
    n_files = len(data[tf])
    print(f'{tf}: {n_files} files loaded')

## EDA 1 — strat-candle transition predictability per TF

Heatmap: rows = prev_strat_candle, cols = curr_strat_candle, color = mean fwd-bar return (bps). Brighter color = bigger move. Look for asymmetries — e.g. does 2D→2D continue more than 2U→2U reverses?

In [ ]:
# Cell 3 — strat transition heatmap per TF
fig, axes = plt.subplots(1, len(TFS), figsize=(22, 5), sharey=True)
for ax, tf in zip(axes, TFS):
    df = data[tf].get('01_strat_transition.csv')
    if df is None or df.empty:
        ax.set_title(f'{tf} — no data'); continue
    # Pool across tickers (could split if desired)
    pooled = df.groupby(['prev_strat_candle','strat_candle']).apply(
        lambda g: (g['mean_bps'] * g['n']).sum() / g['n'].sum()
    ).reset_index().rename(columns={0:'mean_bps'})
    pivot = pooled.pivot(index='prev_strat_candle', columns='strat_candle', values='mean_bps')
    sns.heatmap(pivot, annot=True, fmt='.1f', center=0, cmap='RdBu_r', ax=ax, cbar=ax==axes[-1])
    ax.set_title(f'{tf}: prev × curr → fwd 5bars bps (pooled)')
plt.tight_layout(); plt.show()

In [ ]:
# Cell 4 — same view but hit_pct (% bars with fwd > 0)
fig, axes = plt.subplots(1, len(TFS), figsize=(22, 5), sharey=True)
for ax, tf in zip(axes, TFS):
    df = data[tf].get('01_strat_transition.csv')
    if df is None or df.empty:
        ax.set_title(f'{tf} — no data'); continue
    pooled = df.groupby(['prev_strat_candle','strat_candle']).apply(
        lambda g: (g['hit_pct'] * g['n']).sum() / g['n'].sum()
    ).reset_index().rename(columns={0:'hit_pct'})
    pivot = pooled.pivot(index='prev_strat_candle', columns='strat_candle', values='hit_pct')
    sns.heatmap(pivot, annot=True, fmt='.1f', center=50, cmap='RdBu_r', vmin=40, vmax=60, ax=ax, cbar=ax==axes[-1])
    ax.set_title(f'{tf}: prev × curr → hit_pct')
plt.tight_layout(); plt.show()

## EDA 2 — Strat combo × dealer regime (GEX × VEX) heatmap

This is THE central plot for the user's question. For each (strat_combo × dealer_regime), what's the fwd-return hit rate? Brighter cells = stronger edge. Look for cells where the strat-combo's edge is concentrated in one specific GEX/VEX regime.

In [ ]:
# Cell 5 — combo × dealer_regime heatmap, per TF, per ticker (top-N combos by N)
def plot_combo_regime(tf, ticker='QQQ', top_n_combos=10):
    df = data[tf].get('02_combo_dealer_regime.csv')
    if df is None or df.empty:
        print(f'{tf} {ticker}: no data'); return
    sub = df[df['ticker']==ticker].copy()
    # Take top combos by total N
    top_combos = sub.groupby('strat_combo')['n'].sum().nlargest(top_n_combos).index
    sub = sub[sub['strat_combo'].isin(top_combos)]
    pivot = sub.pivot_table(index='strat_combo', columns='dealer_regime', values='hit_pct', aggfunc='first')
    if pivot.empty:
        print(f'{tf} {ticker}: pivot empty'); return
    plt.figure(figsize=(14, 8))
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdBu_r', center=50, vmin=40, vmax=60)
    plt.title(f'{ticker} {tf}: hit_pct by strat_combo × dealer_regime')
    plt.tight_layout(); plt.show()

for tf in ['5m', '15m', '60m']:
    plot_combo_regime(tf, 'QQQ')
    plot_combo_regime(tf, 'IWM')

## EDA 3 — Indicator correlation with fwd-return, by strat combo

For each strat_combo, which indicators are most correlated with the next-bar return? Use Spearman (rank correlation) — robust to outliers.

In [ ]:
# Cell 6 — top indicator correlations per combo (5m TF, QQQ)
tf = '5m'; ticker = 'QQQ'
df = data[tf].get('04_indicator_correlations.csv')
if df is not None and not df.empty:
    sub = df[df['ticker']==ticker]
    # Top 10 (feature, combo) pairs by |spearman|
    sub['abs_spear'] = sub['spearman'].abs()
    top = sub.nlargest(20, 'abs_spear')[['strat_combo', 'feature', 'n', 'pearson', 'spearman']]
    print(f'Top 20 (feature × combo) by |Spearman| for {ticker} {tf}:')
    display(top)
else:
    print(f'{tf} indicator_correlations not loaded')

In [ ]:
# Cell 7 — heatmap of all indicators × top-N combos, |Spearman|
tf = '5m'; ticker = 'QQQ'
df = data[tf].get('04_indicator_correlations.csv')
if df is not None and not df.empty:
    sub = df[df['ticker']==ticker].copy()
    # Take top 10 most-frequent combos
    top_combos = sub.groupby('strat_combo')['n'].sum().nlargest(10).index
    sub = sub[sub['strat_combo'].isin(top_combos)]
    pivot = sub.pivot_table(index='feature', columns='strat_combo', values='spearman', aggfunc='first')
    plt.figure(figsize=(14, 12))
    sns.heatmap(pivot, annot=False, cmap='RdBu_r', center=0, vmin=-0.15, vmax=0.15)
    plt.title(f'{ticker} {tf}: indicator × strat_combo Spearman correlation with fwd_5bars')
    plt.tight_layout(); plt.show()

## EDA 4 — ML model walk-forward results across TFs

Compare Ridge / Lasso / LightGBM IC across the 5 timeframes. Higher TF should have lower noise / higher IC if there's signal. If LightGBM ≫ linear models, signal is non-linear. If linear ≫ LightGBM, signal is mostly linear (trees overfit).

In [ ]:
# Cell 8 — IC by model × TF
summary_rows = []
for tf in TFS:
    s = data[tf].get('05b_model_summary.csv')
    if s is None: continue
    s = s.set_index('model')
    for model in s.index:
        summary_rows.append({'tf': tf, 'model': model,
                              'mean_ic': s.loc[model, 'mean_ic'],
                              'mean_rank_ic': s.loc[model, 'mean_rank_ic'],
                              'mean_ls_sharpe': s.loc[model, 'mean_ls_sharpe'],
                              'mean_ls_bps': s.loc[model, 'mean_ls_bps']})
summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    print('Model summary across TFs:')
    display(summary_df.pivot(index='tf', columns='model', values='mean_ic'))

    # IC bar chart
    fig, ax = plt.subplots(figsize=(12, 5))
    summary_df.pivot(index='tf', columns='model', values='mean_ic').reindex(TFS).plot.bar(ax=ax)
    ax.axhline(0, color='k', lw=0.5)
    ax.set_title('Mean IC by model × TF (purged walk-forward CV)')
    ax.set_ylabel('Mean IC')
    plt.tight_layout(); plt.show()

    # Cost-adjusted LS Sharpe
    fig, ax = plt.subplots(figsize=(12, 5))
    summary_df.pivot(index='tf', columns='model', values='mean_ls_sharpe').reindex(TFS).plot.bar(ax=ax)
    ax.axhline(0, color='k', lw=0.5)
    ax.set_title('Mean L/S Sharpe (after 5 bps/leg costs) by model × TF')
    ax.set_ylabel('Annualized Sharpe')
    plt.tight_layout(); plt.show()

In [ ]:
# Cell 9 — fold-level IC distribution per TF (shows regime stability)
fold_rows = []
for tf in TFS:
    s = data[tf].get('05a_model_walkforward.csv')
    if s is None: continue
    s = s.copy(); s['tf'] = tf
    fold_rows.append(s)
if fold_rows:
    folds_df = pd.concat(fold_rows, ignore_index=True)
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.boxplot(data=folds_df, x='tf', y='ic', hue='model', ax=ax, order=TFS)
    ax.axhline(0, color='k', lw=0.5)
    ax.set_title('Per-fold IC distribution by model × TF (5 folds)')
    plt.tight_layout(); plt.show()

## EDA 5 — Top features by LightGBM gain importance, per TF

Caveats: tree gain ≠ true predictive importance, but it shows what the tree spends its split-budget on. If VIX-derived features dominate every TF, the signal is mostly vol-regime.

In [ ]:
# Cell 10 — top 15 features by gain, per TF
fig, axes = plt.subplots(1, len(TFS), figsize=(22, 8), sharey=False)
for ax, tf in zip(axes, TFS):
    df = data[tf].get('05c_feature_importance_top50.csv')
    if df is None or df.empty:
        ax.set_title(f'{tf} — no data'); continue
    top15 = df.head(15)
    ax.barh(top15['feature'][::-1], top15['gain'][::-1])
    ax.set_title(f'{tf}: top 15 features by LGBM gain')
    ax.tick_params(axis='y', labelsize=8)
plt.tight_layout(); plt.show()

## EDA 6 — Sanity: bar count + coverage by TF / ticker

Quick check that all 3 ETFs × 5 TFs are populated with expected row counts.

In [ ]:
# Cell 11 — query Cloud SQL directly via SQLAlchemy/Cloud SQL Auth Proxy
# (only works if running this notebook in a place with DB access — locally
#  the sandbox blocks port 5432; use db-query.yml workflow instead).
# This cell is a SKELETON — uncomment + adapt as needed.

# from gcp.database import get_engine
# engine = get_engine()
# rows = pd.read_sql('''
#     SELECT 'strat_features_1m' AS t, ticker, count(*) AS n FROM strat_features_1m GROUP BY ticker
#     UNION ALL
#     SELECT 'strat_features_5m', ticker, count(*) FROM strat_features_5m GROUP BY ticker
#     UNION ALL
#     SELECT 'strat_features_15m', ticker, count(*) FROM strat_features_15m GROUP BY ticker
#     UNION ALL
#     SELECT 'strat_features_30m', ticker, count(*) FROM strat_features_30m GROUP BY ticker
#     UNION ALL
#     SELECT 'strat_features_60m', ticker, count(*) FROM strat_features_60m GROUP BY ticker
#     ORDER BY t, ticker
# ''', engine)
# display(rows.pivot(index='t', columns='ticker', values='n'))
print('Coverage check skeleton — see comment')

## EDA 7 — Reverify prior audit findings

Use the new dataset to check the 4 priority-1 findings from P6:

1. `212_bear_continuation × HIGH-VIX, 5d = +5.15pp` (P3)
2. `clean_2d_bear × HIGH-VIX, 5d = +5.05pp` (P3)
3. `gate_break PUT × LOW-VIX, 1d = -6.4pp` (P5)
4. `322_bull_continuation, 5d = -2.79pp` (P3, anti-predictive)

For each: pull from the new `strat_features_*` tables, compute the same metric, compare. Note these P2/P3/P5 numbers were on DAILY bars; the per-TF tables have INTRADAY bars, so the apples-to-apples comparison uses fwd-return at the daily-equivalent horizon (e.g. 60m × 7 = ~7 hours = the next day's morning bars).

In [ ]:
# Cell 12 — reverify finding 1: 212_bear_continuation × HIGH-VIX at 60m TF
tf = '60m'
df = data[tf].get('03a_combo_vix.csv')
if df is not None:
    cell = df[(df['strat_combo']=='212_bear_continuation') & (df['vix_tercile']=='HIGH')]
    print('212_bear_continuation × HIGH-VIX (60m TF, fwd_5bars=5h):')
    display(cell)
    
    cell2 = df[(df['strat_combo']=='clean_2d_bear') & (df['vix_tercile']=='HIGH')]
    print('clean_2d_bear × HIGH-VIX (60m TF):')
    display(cell2)

## Next steps

After running through this notebook, the next stages of Phase 7 are:

- `scripts/research/p7_compile_report.py` — auto-generates `P7_multi_tf_dataset.md` from these CSVs
- `scripts/research/p7_reverify.py` — produces the formal P7_reverify_prior_findings.md table

The dataset itself (`strat_features_*` in Cloud SQL) is the reusable foundation
for any future research question that needs bar-level strat sequences.